In [62]:
# for correct relative imports
import sys; sys.path.append("../var_es_dgm")

In [63]:
import warnings
warnings.filterwarnings("ignore")

import torch
from torch.utils.data import DataLoader, TensorDataset

from diffusers import DDPMScheduler
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pprint
import scipy.stats

from sklearn.preprocessing import StandardScaler

# Импорт моделей
from var_es_dgm import TimeGrad
from var_es_dgm.basic_models.deepar import DeepAR               
from var_es_dgm.basic_models.deepvar import DeepVaR
from var_es_dgm.basic_models import HistoricalSimulation, VarCov
from var_es_dgm.stat_tests import generate_report
from var_es_dgm.utils import seed_everything, compute_individual_returns, compute_portfolio_returns, estimate_var_es_torch_multivariate

# Импорт GARCH
from arch import arch_model

In [64]:
sns.set_style("darkgrid", {"axes.facecolor": ".9"})
sns.set_context("paper")

In [65]:
DATA_FOLDER = "../../data/"
df = pd.read_csv(DATA_FOLDER + "complete_stocks.csv")
df["Date"] = pd.to_datetime(df["Date"])

In [66]:
len(df["Ticker"].unique())

89

In [67]:
res_timeGrad = list()
res_deepar_normal = list()   
res_deepar_student = list()  
res_deepvar_paper = list()   
res_garch = list()           
res_hist = list()
res_varcov = list()
alpha = 0.05

In [68]:
RANDOM_STATE = 12

In [69]:
for i in range(5):
    # one complete cycle
    seed_everything(RANDOM_STATE + i)
    n_stocks = 10
    tickers = np.random.choice(df["Ticker"].unique(), n_stocks, replace=False)
    weights = 1/n_stocks
    portfolio_weights_array = np.full(n_stocks, weights) 
    print("Portfolio = {0}".format(" + ".join([f"{weights} * {i}"for i in tickers])))
    df_copy = df.loc[df["Ticker"].isin(tickers)].copy(deep=True)

    df_returns = compute_individual_returns(df_copy)
    df_returns = compute_portfolio_returns(df_returns)

    return_cols = [i for i in df_returns.columns if (i.startswith("Return_") and i != "Return_Target")]
    multivariate_returns = df_returns[return_cols]
    multivariate_target = df_returns["Return_Target"]

    multivariate_target = multivariate_target.values[1:]
    train_size = df_returns[df_returns.Date <= "2022-06-01"].index[-1] + 1
    test_size = len(multivariate_target) - train_size
    train = multivariate_returns.values[1:train_size]

    ss = StandardScaler()
    train_scaled = torch.tensor(ss.fit_transform(train), dtype=torch.float32)

    context_size = 90
    num_train_samples = 3000
    train_data = torch.zeros(num_train_samples, context_size, train_scaled.shape[1])
    train_target = torch.zeros(num_train_samples, 1, train_scaled.shape[1])
    train_idx = np.random.choice(np.arange(context_size, train_scaled.shape[0]), num_train_samples, replace=False)

    for i in tqdm(range(num_train_samples)):
        idx = train_idx[i]
        train_context = train_scaled[idx-context_size:idx]
        target_obs = train_scaled[idx]
        train_data[i] = train_context
        train_target[i] = target_obs
    
    # Create DataLoader for ease of torch training
    train_loader = DataLoader(TensorDataset(train_data, train_target), batch_size=128, shuffle=False)


    temp = torch.tensor(ss.transform(multivariate_returns.values[1:]))
    test_data_context = torch.zeros(test_size, context_size, temp.shape[1])
    test_data_real = torch.zeros(test_size, 1, 1)
    for i in range(test_size):
        idx = i + train_size
        test_data_context[i] = temp[idx-context_size:idx]
        test_data_real[i] = multivariate_target[idx]
    
    sheduler = DDPMScheduler(num_train_timesteps=12, beta_end=0.022429104089340533, clip_sample=False)
    model = TimeGrad(train.shape[-1], train.shape[-1], hidden_size=50, num_layers=2, scheduler=sheduler, num_inference_steps=12)
    
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0030588361272846074)
    device = "cuda"
    n_epochs = 96
    model.to(device);

    model.fit(train_loader, optimizer, n_epochs, device)


    VaR_TimeGrad = torch.zeros(test_data_real.shape[0])
    ES_TimeGrad = torch.zeros(test_data_real.shape[0])
    
    for i in tqdm(range(test_data_real.shape[0])):
        test = test_data_context[[i]]
        
        # compute correlation matrix
        pho = torch.corrcoef(torch.squeeze(test).T).to(torch.double)
        
        VaR_TimeGrad[i], ES_TimeGrad[i] = estimate_var_es_torch_multivariate(model, test, ss, pho, alpha=alpha, n_samples=500, device=device)

    res_timeGrad.append(generate_report(test_data_real.flatten(), VaR_TimeGrad, ES_TimeGrad, alpha=alpha))

     # ==========================================
    # 2. DeepAR (Normal)
    # ==========================================

    n_epochs = 20
    model_deepar_normal = DeepAR(target_dim=train.shape[-1], input_size=train.shape[-1], hidden_size=64, num_layers=2, dist_type='normal')
    optimizer_dar_n = torch.optim.Adam(model_deepar_normal.parameters(), lr=1e-3)
    model_deepar_normal.to(device)
    model_deepar_normal.fit(train_loader, optimizer_dar_n, n_epochs=n_epochs, device=device)

    VaR_DeepAR_N = torch.zeros(test_size)
    ES_DeepAR_N = torch.zeros(test_size)
    for j in tqdm(range(test_size), desc="Testing DeepAR (Normal)"):
        test = test_data_context[[j]].to(device)
        pho = torch.corrcoef(torch.squeeze(test).T).to(torch.double)
        VaR_DeepAR_N[j], ES_DeepAR_N[j] = estimate_var_es_torch_multivariate(model_deepar_normal, test, ss, pho, alpha=alpha, n_samples=500, device=device)
    res_deepar_normal.append(generate_report(test_data_real.flatten(), VaR_DeepAR_N, ES_DeepAR_N, alpha=alpha))

    # ==========================================
    # 3. DeepAR (Student)
    # ==========================================
    model_deepar_student = DeepAR(target_dim=train.shape[-1], input_size=train.shape[-1], hidden_size=64, num_layers=2, dist_type='student')
    optimizer_dar_s = torch.optim.Adam(model_deepar_student.parameters(), lr=1e-3)
    model_deepar_student.to(device)
    model_deepar_student.fit(train_loader, optimizer_dar_s, n_epochs=n_epochs, device=device)

    VaR_DeepAR_S = torch.zeros(test_size)
    ES_DeepAR_S = torch.zeros(test_size)
    for j in tqdm(range(test_size), desc="Testing DeepAR (Student)"):
        test = test_data_context[[j]].to(device)
        pho = torch.corrcoef(torch.squeeze(test).T).to(torch.double)
        VaR_DeepAR_S[j], ES_DeepAR_S[j] = estimate_var_es_torch_multivariate(model_deepar_student, test, ss, pho, alpha=alpha, n_samples=500, device=device)
    res_deepar_student.append(generate_report(test_data_real.flatten(), VaR_DeepAR_S, ES_DeepAR_S, alpha=alpha))

    # ==========================================
    # 4. DeepVaR Framework (Логика из статьи)
    # ==========================================
    deepvar_algo = DeepVaR(model=model_deepar_student, scaler=ss, alpha=alpha, n_samples=500, device=device, unscale_predictions = True)
    
    VaR_DeepVaR_paper = torch.zeros(test_size)
    ES_DeepVaR_paper = torch.zeros(test_size)
    
    for j in tqdm(range(test_size), desc="Testing DeepVaR (Framework)"):
        test = test_data_context[[j]].to(device)
        # Рассчитываем матрицу корреляций по контекстному окну (как в вашем базовом коде)
        pho_tensor = torch.corrcoef(torch.squeeze(test).T).to(torch.float32)
        
        v, e = deepvar_algo.predict(test, weights=portfolio_weights_array, corr_matrix=pho_tensor)
        VaR_DeepVaR_paper[j] = v
        ES_DeepVaR_paper[j] = e
        
    res_deepvar_paper.append(generate_report(test_data_real.flatten(), VaR_DeepVaR_paper, ES_DeepVaR_paper, alpha=alpha))

    # ==========================================
    # 5. GARCH(1,1) Model
    # ==========================================
    VaR_garch = torch.zeros(test_size)
    ES_garch = torch.zeros(test_size)
    
    for j in tqdm(range(test_size), desc="Testing GARCH(1,1)"):
        idx = j + train_size
        start_idx = max(0, idx - 900)
        hist_returns = multivariate_target[start_idx:idx]
        
        am = arch_model(hist_returns * 100, vol='Garch', p=1, q=1, dist='Normal')
        res = am.fit(disp='off', show_warning=False)
        forecasts = res.forecast(horizon=1)
        
        mu = forecasts.mean.iloc[-1, 0] / 100
        sigma = np.sqrt(forecasts.variance.iloc[-1, 0]) / 100
        
        q = scipy.stats.norm.ppf(alpha)
        VaR_garch[j] = (mu + sigma * q)
        pdf_q = scipy.stats.norm.pdf(q)
        ES_garch[j] = (mu - sigma * (pdf_q / alpha))

    res_garch.append(generate_report(test_data_real.flatten(), VaR_garch, ES_garch, alpha=alpha))

    hist_sim = HistoricalSimulation(alpha=alpha)

    VaR_histSim = torch.zeros(test_data_real.shape[0])
    ES_histSim = torch.zeros(test_data_real.shape[0])
    
    for i in tqdm(range(test_data_real.shape[0])):
        test = test_data_context[[i]]
        VaR_histSim[i], ES_histSim[i] = hist_sim.predict(test_data_context[[i]], scaler=ss)
    
    res_hist.append(generate_report(test_data_real.flatten(), VaR_histSim, ES_histSim, alpha=alpha))
    
    var_cov = VarCov(alpha=alpha)
    
    VaR_varCov = torch.zeros(test_data_real.shape[0])
    ES_varCov = torch.zeros(test_data_real.shape[0])
    
    for i in tqdm(range(test_data_real.shape[0])):
        test = test_data_context[[i]]
        VaR_varCov[i], ES_varCov[i] = var_cov.predict(test_data_context[[i]], scaler=ss)
    
    res_varcov.append(generate_report(test_data_real.flatten(), VaR_varCov, ES_varCov, alpha=alpha))

    print("TimeGrad:      ", res_timeGrad[-1])
    print("DeepAR Normal: ", res_deepar_normal[-1])
    print("DeepAR Student:", res_deepar_student[-1])
    print("DeepVaR Paper: ", res_deepvar_paper[-1])
    print("GARCH(1,1):    ", res_garch[-1])
    print("HistSim:       ", res_hist[-1])
    print("VarCov:        ", res_varcov[-1])

Portfolio = 0.1 * INTC + 0.1 * DHR + 0.1 * COP + 0.1 * MRK + 0.1 * HON + 0.1 * T + 0.1 * MSFT + 0.1 * AXP + 0.1 * PEP + 0.1 * SYK


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

Epochs:   0%|          | 0/96 [00:00<?, ?it/s]

  0%|          | 0/396 [00:00<?, ?it/s]

...

Testing DeepVaR (Framework):   0%|          | 0/396 [00:00<?, ?it/s]

Testing GARCH(1,1):   0%|          | 0/396 [00:00<?, ?it/s]

  0%|          | 0/396 [00:00<?, ?it/s]

  0%|          | 0/396 [00:00<?, ?it/s]

TimeGrad:       {'Kupicks POF': 0.3653929983513471, 'Haas TBF': 0.054769558027543595, 'Christoffersen Test': 0.5415791828627718, 'Kupicks Test': 0.3653929983513471, 'Acerbi Szekely 1': 2.7434895038604736, 'Acerbi Szekely 2': 0.00554240308701992}
DeepAR Normal:  {'Kupicks POF': 0.11478649504904415, 'Haas TBF': 0.06426908708865625, 'Christoffersen Test': 0.4444870465584825, 'Kupicks Test': 0.11478649504904415, 'Acerbi Szekely 1': 2.5920584201812744, 'Acerbi Szekely 2': 0.008836562745273113}
DeepAR Student: {'Kupicks POF': 0.005273537988593865, 'Haas TBF': 0.024361378909931283, 'Christoffersen Test': 0.41313554781681816, 'Kupicks Test': 0.005273537988593865, 'Acerbi Szekely 1': 2.5705935955047607, 'Acerbi Szekely 2': 0.010710807517170906}
DeepVaR Paper:  {'Kupicks POF': 0.009515381736260306, 'Haas TBF': 0.03987285895570076, 'Christoffersen Test': 0.41896313548347053, 'Kupicks Test': 0.009515381736260306, 'Acerbi Szekely 1': 2.9617326259613037, 'Acerbi Szekely 2': 0.011966596357524395}
GAR

In [70]:
print("Mean TimeGrad:      ", np.array([list(x.values()) for x in res_timeGrad]).mean(axis=0))
print("Mean DeepAR Normal: ", np.array([list(x.values()) for x in res_deepar_normal]).mean(axis=0))
print("Mean DeepAR Student:", np.array([list(x.values()) for x in res_deepar_student]).mean(axis=0))
print("Mean DeepVaR Paper: ", np.array([list(x.values()) for x in res_deepvar_paper]).mean(axis=0))
print("Mean GARCH(1,1):    ", np.array([list(x.values()) for x in res_garch]).mean(axis=0))
print("Mean HistSim:       ", np.array([list(x.values()) for x in res_hist]).mean(axis=0))
print("Mean VarCov:        ", np.array([list(x.values()) for x in res_varcov]).mean(axis=0))

Mean TimeGrad:       [0.32053673 0.08400466 0.53430777 0.32053673 2.72631793 0.00600092]
Mean DeepAR Normal:  [0.33345148 0.25913801 0.50666515 0.33345148 2.5900681  0.0063574 ]
Mean DeepAR Student: [0.26494131 0.16726269 0.45381116 0.26494131 2.56368585 0.00857108]
Mean DeepVaR Paper:  [0.29545744 0.14474623 0.46479452 0.29545744 3.06793475 0.00981404]
Mean GARCH(1,1):     [0.26790727 0.39737836 0.52973799 0.26790727 2.10345545 0.00428543]
Mean HistSim:        [0.45848716 0.03499475 0.47515655 0.45848716 2.40126729 0.00716972]
Mean VarCov:         [0.59423254 0.1037123  0.52292489 0.59423254 2.0204572  0.00444999]
